# 08 · Recursion with matrices and vectors / Recursión con matrices y vectores

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/08-recursion-with-matrices.ipynb)

*Part IV · demo · 10 min*

This notebook is about one simple pattern:

> **Use the current state to create the next state, then repeat.**

We will see that same pattern in:

1. Fibonacci numbers,
2. power iteration,
3. a real monthly airline-passenger forecast.

> 🇪🇸 Este cuaderno estudia un patrón muy sencillo:
>
> **Usar el estado actual para crear el siguiente estado y repetir el proceso.**
>
> Veremos la misma idea en:
>
> 1. números de Fibonacci,
> 2. iteración de potencias,
> 3. un pronóstico real de pasajeros mensuales de aerolíneas.

## What you will be able to do / Lo que podrás hacer

- Explain **recursion / recurrence**, **state**, and **state update** in plain language.
- Write `x[t+1] = A @ x[t]` and explain what every symbol means.
- Turn Fibonacci into a two-number state updated by the same matrix.
- See why repeated multiplication can align a vector with a dominant eigenvector.
- Understand why the ratio `|λ₂/λ₁|` affects convergence speed.
- Fit a real autoregressive model with the pseudoinverse.
- Compare a recursive forecast with a one-step diagnostic that uses real previous values.

> 🇪🇸
>
> - Explicar **recursión / recurrencia**, **estado** y **actualización de estado** en lenguaje sencillo.
> - Escribir `x[t+1] = A @ x[t]` y explicar qué significa cada símbolo.
> - Convertir Fibonacci en un estado de dos números actualizado por la misma matriz.
> - Observar por qué multiplicar repetidamente puede alinear un vector con un autovector dominante.
> - Entender por qué la razón `|λ₂/λ₁|` afecta la velocidad de convergencia.
> - Ajustar un modelo autorregresivo real con la pseudoinversa.
> - Comparar un pronóstico recursivo con un diagnóstico de un paso que usa valores previos reales.

## Start with an everyday analogy / Empecemos con una analogía cotidiana

Imagine a bank account.

Today's balance becomes part of the calculation for tomorrow's balance.

Then tomorrow's balance becomes part of the calculation for the next day.

That is the idea of a **recurrence**:

**the output from one step becomes information used in the next step.**

### Three useful words / Tres palabras útiles

| Term / Término | Plain meaning / Significado sencillo |
|---|---|
| **State / Estado** | the information we carry from one step to the next / la información que llevamos de un paso al siguiente |
| **Update rule / Regla de actualización** | the recipe that converts the current state into the next one / la receta que convierte el estado actual en el siguiente |
| **Recursion / Recurrence / Recursión / Recurrencia** | applying that update repeatedly / aplicar esa actualización repetidamente |

In this notebook, the update rule often looks like:

`x[t+1] = A @ x[t]`

Read it as:

> **next state = same matrix × current state**

> 🇪🇸 Léelo como:
>
> **estado siguiente = la misma matriz × estado actual**

## Read the recurrence as a sentence / Lee la recurrencia como una frase

For:

`x[t+1] = A @ x[t]`

- `t` = current step / paso actual
- `x[t]` = current state / estado actual
- `A` = update rule written as a matrix / regla de actualización escrita como matriz
- `x[t+1]` = next state / estado siguiente

The matrix `A` stays the same in the simplest examples. The state changes at every step.

> 🇪🇸 En los ejemplos más sencillos, la matriz `A` permanece igual. Lo que cambia en cada paso es el estado.

## Setup / Preparación

Run this cell first.

It loads the real monthly airline-passenger dataset used later and the small visualization tools used throughout the notebook.

> 🇪🇸 Ejecuta primero esta celda.
>
> Carga el conjunto real de pasajeros mensuales de aerolíneas que usaremos más adelante y las herramientas de visualización del cuaderno.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

FLIGHTS = (
    "https://raw.githubusercontent.com/mwaskom/"
    "seaborn-data/master/flights.csv"
)

flights = pd.read_csv(FLIGHTS)

y = flights["passengers"].to_numpy(float)

labels = (
    flights["year"].astype(str)
    + "-"
    + flights["month"].astype(str).str[:3]
).to_numpy()

rng = np.random.default_rng(0)

print("Real months / Meses reales:", len(y))
print("Range / Periodo:", labels[0], "→", labels[-1])
print("Passengers min/max / Pasajeros mín/máx:", int(y.min()), int(y.max()))
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## Why this matters / Por qué esto importa

The same recursive skeleton can have very different consequences.

### Fibonacci
Repeated updates generate a sequence exactly.

### Power iteration
Repeated multiplication can reveal a dominant direction in a matrix.

### Recursive forecasting
A prediction becomes an input to the next prediction. If one prediction is wrong, part of that error can travel forward.

So repetition can:

- **build structure**, or
- **propagate error**.

### Learning cycle / Ciclo de aprendizaje

Use:

**Predict → Run → Explain / Predice → Ejecuta → Explica**

Before each exercise, ask:

1. What is the state?
2. What is the update rule?
3. What information is carried to the next step?
4. What could repeated updates amplify?

> 🇪🇸 La repetición puede construir estructura o propagar errores.
>
> Antes de cada ejercicio pregunta:
>
> 1. ¿Cuál es el estado?
> 2. ¿Cuál es la regla de actualización?
> 3. ¿Qué información pasa al siguiente paso?
> 4. ¿Qué podría amplificarse al repetir?

### Interactive state-update translator / Traductor interactivo de actualización de estado

Choose one example and identify its state and update rule.

> 🇪🇸 Elige un ejemplo e identifica su estado y su regla de actualización.

In [ ]:
state_example = widgets.Dropdown(
    options=[
        ("Fibonacci", "fib"),
        ("Power iteration / Iteración de potencias", "power"),
        ("Recursive forecast / Pronóstico recursivo", "forecast"),
    ],
    value="fib",
    description="Example / Ejemplo:",
    style={"description_width": "120px"},
)

def explain_state_example(example):
    examples = {
        "fib": (
            "[f[n], f[n-1]]",
            "multiply by the same 2×2 Fibonacci matrix",
            "multiplicar por la misma matriz Fibonacci 2×2",
            "the two most recent sequence values",
            "los dos valores más recientes de la secuencia",
        ),
        "power": (
            "current direction vector x[t]",
            "multiply by A, then normalize",
            "multiplicar por A y después normalizar",
            "the current estimate of the dominant direction",
            "la estimación actual de la dirección dominante",
        ),
        "forecast": (
            "recent passenger history",
            "predict next month, append prediction, repeat",
            "predecir el mes siguiente, agregar la predicción y repetir",
            "the recent values used to predict the future",
            "los valores recientes usados para predecir el futuro",
        ),
    }

    state, update_en, update_es, carry_en, carry_es = examples[example]

    print("State / Estado:", state)
    print("Update EN:", update_en)
    print("Actualización ES:", update_es)
    print("Carries EN:", carry_en)
    print("Transporta ES:", carry_es)

state_output = widgets.interactive_output(
    explain_state_example,
    {"example": state_example},
)

display(widgets.VBox([state_example, state_output]))

## Exercise 1 — Fibonacci as a matrix state / Ejercicio 1 — Fibonacci como estado matricial

Fibonacci is usually written:

`f[n+1] = f[n] + f[n-1]`

This looks like a scalar recurrence.

But the next value needs **two pieces of memory**:

- the current Fibonacci number;
- the previous Fibonacci number.

So we store both in one state:

`[f[n], f[n-1]]`

and update it with:

`F = [[1,1],[1,0]]`

### Why does this work? / ¿Por qué funciona?

The first row says:

`new first value = current + previous`

The second row says:

`new second value = old current`

That means the state automatically shifts forward.

> 🇪🇸 La primera fila suma los dos valores para crear el siguiente Fibonacci.
>
> La segunda fila copia el valor actual a la posición de “anterior”.
>
> Así el estado avanza automáticamente.

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Define:
#       F = [[1, 1],
#            [1, 0]]
# 2. Start with state [1, 0].
# 3. Apply F repeatedly for 10 steps.
# 4. Print every state.
# 5. Compare the final result with:
#       np.linalg.matrix_power(F, 10) @ v0
#
# ES:
# 1. Define la matriz F.
# 2. Empieza con el estado [1, 0].
# 3. Aplica F repetidamente durante 10 pasos.
# 4. Imprime cada estado.
# 5. Compara el resultado final con matrix_power.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

F = np.array(
    [[1, 1],
     [1, 0]],
    dtype=object,
)

v0 = np.array([1, 0], dtype=object)

states = [v0.copy()]
v = v0.copy()

for _ in range(10):
    v = F @ v
    states.append(v.copy())

via_power = np.linalg.matrix_power(F, 10) @ v0

print("Step / Paso | State / Estado")
for k, state in enumerate(states):
    print(f"{k:>4}        {state.tolist()}")

print()
print("Loop final / Final del ciclo:", v.tolist())
print("matrix_power:", via_power.tolist())
print("Same result / Mismo resultado:", np.array_equal(v, via_power))
print("Fibonacci(10):", int(v[1]))
print()
print("EN: matrix_power compresses ten repeated updates into one matrix power.")
print("ES: matrix_power comprime diez actualizaciones repetidas en una sola potencia matricial.")

### Interactive Fibonacci state explorer / Explorador interactivo del estado Fibonacci

Move **Steps / Pasos** or press **Play**.

Watch the two-number state:

`[next value, current value]`

grow after every repeated multiplication.

> 🇪🇸 Mueve **Pasos** o presiona **Play**.
>
> Observa cómo crece el estado de dos números:
>
> `[valor siguiente, valor actual]`

In [ ]:
fib_steps = widgets.IntSlider(
    value=10,
    min=1,
    max=25,
    step=1,
    description="Steps / Pasos:",
    continuous_update=False,
    style={"description_width": "100px"},
)

fib_play = widgets.Play(
    value=10,
    min=1,
    max=25,
    step=1,
    interval=600,
    description="Play",
)

# IMPORTANT FOR COLAB:
# widgets.jslink() only links values in the browser.
# The graph is redrawn by Python, so we use kernel-side widgets.link().
fib_link = widgets.link(
    (fib_play, "value"),
    (fib_steps, "value"),
)

def explore_fibonacci(steps):
    seq = []
    state = v0.copy()

    for k in range(steps + 1):
        seq.append((k, int(state[0]), int(state[1])))
        state = F @ state

    df = pd.DataFrame(
        seq,
        columns=["step", "next_value", "current_value"],
    )

    final_state = np.linalg.matrix_power(F, steps) @ v0

    plt.close("all")
    fig, ax = plt.subplots(
        figsize=(8.5, 4.2),
        constrained_layout=True,
    )

    ax.plot(
        df["step"],
        df["next_value"],
        marker="o",
        label="next value / valor siguiente",
    )
    ax.plot(
        df["step"],
        df["current_value"],
        marker="o",
        label="current value / valor actual",
    )

    # Keep the x-axis stable so the animation feels like the curve is growing.
    ax.set_xlim(0, fib_steps.max)

    # Use the largest value visible at this step, with a small margin.
    ymax = max(
        1,
        int(df[["next_value", "current_value"]].to_numpy().max())
    )
    ax.set_ylim(0, ymax * 1.12)

    ax.set_xlabel("step / paso")
    ax.set_ylabel("state value / valor del estado")
    ax.set_title(
        f"Fibonacci recursion / Recursión Fibonacci — "
        f"step/paso {steps}"
    )
    ax.legend(loc="upper left")

    plt.show()

    print("Current step / Paso actual:", steps)
    print("Current state / Estado actual:", final_state.tolist())
    print(f"Fibonacci({steps}) =", int(final_state[1]))
    print("EN: Play advances the state one update at a time.")
    print("ES: Play avanza el estado una actualización a la vez.")

# Observe the Play widget directly.
# Moving the slider also updates Play because widgets.link is bidirectional.
fib_output = widgets.interactive_output(
    explore_fibonacci,
    {"steps": fib_play},
)

display(
    widgets.VBox([
        widgets.HBox([fib_play, fib_steps]),
        fib_output,
    ])
)

### See one update numerically / Observa una actualización numéricamente

Choose a step. The notebook shows the state **before** and **after** multiplying by `F`.

> 🇪🇸 Elige un paso. El cuaderno muestra el estado **antes** y **después** de multiplicar por `F`.

In [ ]:
fib_one_step = widgets.IntSlider(
    value=3,
    min=0,
    max=15,
    step=1,
    description="Step / Paso:",
    continuous_update=False,
    style={"description_width": "95px"},
)

def explain_fib_step(step):
    before = np.linalg.matrix_power(F, step) @ v0
    after = F @ before

    print("Before / Antes:", before.tolist())
    print()
    print("F @ state / F @ estado")
    print(F)
    print("@")
    print(before)
    print("=")
    print(after)
    print()
    print(
        f"EN: {int(after[0])} = {int(before[0])} + {int(before[1])}; "
        f"the old current value {int(before[0])} shifts into the second position."
    )
    print(
        f"ES: {int(after[0])} = {int(before[0])} + {int(before[1])}; "
        f"el valor actual anterior {int(before[0])} pasa a la segunda posición."
    )

fib_step_output = widgets.interactive_output(
    explain_fib_step,
    {"step": fib_one_step},
)

display(widgets.VBox([fib_one_step, fib_step_output]))

<details>
<summary><strong>What did Exercise 1 teach? / ¿Qué enseñó el Ejercicio 1?</strong></summary>

A recurrence may look like a sequence of scalar formulas, but we can often collect the necessary memory into a **state vector**.

Then one repeated matrix multiplication advances the whole state.

`matrix_power(F, n)` is useful because:

`Fⁿ @ x[0]`

represents applying the same state update `n` times.

> 🇪🇸 Una recurrencia puede parecer una secuencia de fórmulas escalares, pero podemos reunir la memoria necesaria en un **vector de estado**.
>
> `Fⁿ @ x[0]` representa aplicar la misma actualización `n` veces.

</details>

## Exercise 2 — when repetition chooses a direction / Ejercicio 2 — cuando la repetición elige una dirección

Power iteration also repeats:

`x[t+1] = A @ x[t]`

but after each multiplication we normalize the vector.

For a suitable matrix, the direction tends to align with the eigenvector whose eigenvalue has the largest magnitude.

### What is an eigenvector in simple language? / ¿Qué es un autovector en lenguaje sencillo?

An eigenvector is a direction that a matrix does **not rotate away from itself**.

The matrix can stretch it, shrink it, or reverse its sign, but the direction remains special.

### Dominant eigenvector / Autovector dominante

If one eigenvalue has larger magnitude than the others, repeated multiplication tends to amplify its direction more strongly.

### Why use a synthetic 2×2 matrix here? / ¿Por qué una matriz sintética?

This exercise deliberately uses a controlled synthetic matrix because we want to change one thing cleanly:

`|λ₂ / λ₁|`

and observe how that ratio affects convergence.

This is a mathematical experiment, not a fake real-world dataset.

> 🇪🇸 Un autovector es una dirección especial que la matriz no desvía hacia otra dirección.
>
> Usamos deliberadamente una matriz sintética `2×2` para controlar la razón `|λ₂/λ₁|` y aislar su efecto sobre la velocidad de convergencia.

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Implement power iteration.
# 2. Normalize after every multiplication.
# 3. Compare the current direction with the dominant eigenvector.
# 4. Track the angle between them.
# 5. Compare a small λ₂/λ₁ ratio with a ratio close to 1.
#
# ES:
# 1. Implementa iteración de potencias.
# 2. Normaliza después de cada multiplicación.
# 3. Compara la dirección actual con el autovector dominante.
# 4. Sigue el ángulo entre ambos.
# 5. Compara una razón λ₂/λ₁ pequeña con una cercana a 1.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

def build_controlled_matrix(ratio):
    # Same eigenvectors; eigenvalues are 5 and 5*ratio.
    theta = np.pi / 6

    Q = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)],
    ])

    D = np.diag([5.0, 5.0 * ratio])

    return Q @ D @ Q.T

def dominant_direction(M):
    eigvals, eigvecs = np.linalg.eig(M)

    idx = int(np.argmax(np.abs(eigvals)))

    dominant = eigvecs[:, idx].real
    dominant /= np.linalg.norm(dominant)

    return eigvals, dominant

def power_trace(M, steps=40, seed=0):
    eigvals, dominant = dominant_direction(M)

    v = np.random.default_rng(seed).standard_normal(M.shape[0])
    v /= np.linalg.norm(v)

    rows = []

    for step in range(1, steps + 1):
        v = M @ v
        v /= np.linalg.norm(v)

        alignment = abs(float(np.dot(v, dominant)))
        alignment = np.clip(alignment, 0.0, 1.0)

        angle = np.degrees(np.arccos(alignment))
        rayleigh = float(v @ M @ v)

        rows.append({
            "step": step,
            "angle_deg": angle,
            "rayleigh": rayleigh,
            "x0": v[0],
            "x1": v[1],
        })

    return pd.DataFrame(rows), eigvals, dominant

for ratio in [0.30, 0.95]:
    M = build_controlled_matrix(ratio)
    trace, eigvals, dominant = power_trace(M)

    print("λ₂/λ₁ =", ratio)
    print("Eigenvalues / Autovalores:", np.round(np.sort(eigvals)[::-1], 3))
    print("Final angle / Ángulo final:", f"{trace['angle_deg'].iloc[-1]:.6f}°")
    print()

### Interactive spectral-gap explorer / Explorador interactivo de brecha espectral

Control two things:

- `λ₂/λ₁` — how close the second eigenvalue is to the dominant one;
- number of iterations.

The left graph shows the angle to the dominant eigenvector.

The right graph shows the current direction and the dominant direction.

### Prediction / Predicción

Move `λ₂/λ₁` toward `1`.

What should happen to convergence?

> 🇪🇸 Controla la razón `λ₂/λ₁` y el número de iteraciones.
>
> Acerca la razón a `1` y observa si la convergencia se vuelve más rápida o más lenta.

In [ ]:
ratio_slider = widgets.FloatSlider(
    value=0.40,
    min=0.10,
    max=0.99,
    step=0.01,
    description="λ₂ / λ₁:",
    continuous_update=False,
    readout_format=".2f",
    style={"description_width": "80px"},
)

power_steps = widgets.IntSlider(
    value=12,
    min=1,
    max=40,
    step=1,
    description="Iterations / Iteraciones:",
    continuous_update=False,
    style={"description_width": "145px"},
)

def explore_power_iteration(ratio, steps):
    M = build_controlled_matrix(ratio)
    trace, eigvals, dominant = power_trace(
        M,
        steps=max(steps, 1),
        seed=0,
    )

    current = trace.iloc[-1][["x0", "x1"]].to_numpy(float)
    angle = float(trace["angle_deg"].iloc[-1])

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10.5, 4.0),
        constrained_layout=True,
    )

    axes[0].plot(
        trace["step"],
        trace["angle_deg"],
        marker="o",
    )
    axes[0].set_xlabel("iteration / iteración")
    axes[0].set_ylabel("angle to dominant direction / ángulo")
    axes[0].set_title("Convergence / Convergencia")

    axes[1].quiver(
        [0, 0],
        [0, 0],
        [dominant[0], current[0]],
        [dominant[1], current[1]],
        angles="xy",
        scale_units="xy",
        scale=1,
    )
    axes[1].set_xlim(-1.2, 1.2)
    axes[1].set_ylim(-1.2, 1.2)
    axes[1].axhline(0, linewidth=0.8)
    axes[1].axvline(0, linewidth=0.8)
    axes[1].set_aspect("equal")
    axes[1].set_title(
        "Dominant vs current direction / "
        "Dirección dominante vs actual"
    )

    plt.show()

    print("Eigenvalues / Autovalores:", np.round(np.sort(eigvals)[::-1], 3))
    print("λ₂/λ₁:", f"{ratio:.2f}")
    print("Iteration / Iteración:", steps)
    print("Angle / Ángulo:", f"{angle:.6f}°")
    print()

    if ratio > 0.85:
        print("EN: the eigenvalues are close, so the dominant direction wins slowly.")
        print("ES: los autovalores están cerca, por lo que la dirección dominante se impone lentamente.")
    else:
        print("EN: the spectral gap is larger, so the dominant direction becomes clear faster.")
        print("ES: la brecha espectral es mayor, por lo que la dirección dominante aparece más rápido.")

power_output = widgets.interactive_output(
    explore_power_iteration,
    {
        "ratio": ratio_slider,
        "steps": power_steps,
    },
)

display(
    widgets.VBox([
        widgets.HBox([ratio_slider, power_steps]),
        power_output,
    ])
)

### The important rule / La regla importante

Power iteration converges quickly when the dominant eigenvalue is clearly larger in magnitude.

A useful intuition is:

- small `|λ₂/λ₁|` → faster separation;
- `|λ₂/λ₁|` close to `1` → slower separation.

This is why the **spectral gap** matters.

> 🇪🇸 La iteración de potencias converge más rápido cuando el autovalor dominante es claramente mayor en magnitud.
>
> - `|λ₂/λ₁|` pequeño → separación más rápida;
> - `|λ₂/λ₁|` cercano a `1` → separación más lenta.

## Exercise 3 — recursive forecasting on real airline traffic / Ejercicio 3 — pronóstico recursivo con tráfico aéreo real

Now we use **144 real monthly passenger counts from 1949 to 1960**.

We hold out the final 12 months.

The model sees only the earlier months during fitting.

### Autoregressive model / Modelo autorregresivo

For a window of length `p`:

`next month = bias + w₁·old value + ... + wₚ·recent value`

The coefficients are fitted with the pseudoinverse:

`w = X⁺y`

This directly connects to Notebook 07.

### Why is this recursive? / ¿Por qué es recursivo?

After predicting one month:

1. append that prediction to history;
2. use it as an input for the next prediction;
3. repeat.

So a prediction can influence later predictions.

### Why might `p=12` make sense? / ¿Por qué puede tener sentido `p=12`?

The data are monthly. A 12-month window can contain one full annual seasonal cycle.

That does not guarantee it is the best model, but it gives the model access to one year of recent history.

> 🇪🇸 Usamos 144 observaciones mensuales reales de 1949 a 1960.
>
> Reservamos los últimos 12 meses y ajustamos el modelo solo con los meses anteriores.
>
> Cada predicción se agrega a la historia y se usa para producir la siguiente; por eso el pronóstico es recursivo.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Hold out the final 12 real months.
# 2. Fit an autoregressive model with p=12 using np.linalg.pinv.
# 3. Forecast the 12 held-out months recursively.
# 4. Compute MAPE.
# 5. Try p=3 and p=24.
# 6. Explain why prediction errors can propagate.
#
# ES:
# 1. Reserva los últimos 12 meses reales.
# 2. Ajusta un modelo autorregresivo con p=12 usando np.linalg.pinv.
# 3. Pronostica recursivamente los 12 meses reservados.
# 4. Calcula MAPE.
# 5. Prueba p=3 y p=24.
# 6. Explica por qué los errores de predicción pueden propagarse.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

def fit_ar(series, p):
    rows = np.array(
        [
            series[i:i + p]
            for i in range(len(series) - p)
        ],
        dtype=float,
    )

    X_ar = np.column_stack([
        np.ones(len(rows)),
        rows,
    ])

    target = series[p:]

    return np.linalg.pinv(X_ar) @ target

def recursive_forecast(history, w_ar, p, steps):
    hist = list(
        np.asarray(
            history,
            dtype=float,
        )
    )

    out = []

    for _ in range(steps):
        recent = np.asarray(hist[-p:], dtype=float)

        nxt = float(
            w_ar[0]
            + np.dot(w_ar[1:], recent)
        )

        hist.append(nxt)
        out.append(nxt)

    return np.asarray(out)

def one_step_diagnostic(full_series, train_end, w_ar, p, steps):
    # Diagnostic only:
    # use the real previous observations when predicting each held-out month.
    # This isolates one-step model error from recursive feedback error.
    preds = []

    for t in range(train_end, min(train_end + steps, len(full_series))):
        recent_real = full_series[t - p:t]

        pred = float(
            w_ar[0]
            + np.dot(w_ar[1:], recent_real)
        )

        preds.append(pred)

    return np.asarray(preds)

def mape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    return np.mean(
        np.abs(predicted - actual) / actual
    )

holdout = 12

y_train = y[:-holdout]
y_test = y[-holdout:]

w12 = fit_ar(y_train, 12)

pred12_recursive = recursive_forecast(
    y_train,
    w12,
    12,
    holdout,
)

pred12_one_step = one_step_diagnostic(
    y,
    len(y_train),
    w12,
    12,
    holdout,
)

print("Training months / Meses de entrenamiento:", len(y_train))
print("Held-out months / Meses reservados:", len(y_test))
print(
    "Recursive MAPE / MAPE recursivo:",
    f"{mape(y_test, pred12_recursive):.1%}",
)
print(
    "One-step diagnostic MAPE / MAPE diagnóstico de un paso:",
    f"{mape(y_test, pred12_one_step):.1%}",
)
print()
print("EN: the one-step diagnostic uses real previous held-out values; it is not a deployable recursive forecast.")
print("ES: el diagnóstico de un paso usa valores reales previos del periodo reservado; no es un pronóstico recursivo desplegable.")

### First look at the real series / Primero observa la serie real

Choose how many years of history to display.

Before fitting a model, look for:

- trend;
- repeating yearly pattern;
- increasing seasonal amplitude.

> 🇪🇸 Elige cuántos años de historia quieres ver.
>
> Antes de ajustar un modelo, busca tendencia, patrón anual repetido y cambios en la amplitud estacional.

In [ ]:
years_slider = widgets.IntSlider(
    value=12,
    min=2,
    max=12,
    step=1,
    description="Years / Años:",
    continuous_update=False,
    style={"description_width": "100px"},
)

def show_airline_history(years):
    n = years * 12

    fig, ax = plt.subplots(
        figsize=(10, 3.8),
        constrained_layout=True,
    )

    ax.plot(
        np.arange(n),
        y[:n],
        marker="o",
        markersize=3,
    )

    ax.set_xlabel("month index / índice mensual")
    ax.set_ylabel("passengers / pasajeros")
    ax.set_title(
        f"First {years} years of real airline data / "
        f"Primeros {years} años de datos reales"
    )

    plt.show()

    print("Displayed months / Meses mostrados:", n)
    print("EN: look for trend and annual seasonality before fitting a recurrence.")
    print("ES: busca tendencia y estacionalidad anual antes de ajustar una recurrencia.")

history_output = widgets.interactive_output(
    show_airline_history,
    {"years": years_slider},
)

display(widgets.VBox([years_slider, history_output]))

### Interactive recursive forecast explorer / Explorador interactivo de pronóstico recursivo

Control:

- **Window / Ventana** = how many recent months the model remembers;
- **Horizon / Horizonte** = how many recursive future steps to generate.

For the first 12 forecast steps, we have real held-out values for comparison.

After 12 months, the line moves beyond the available dataset, so there is **no real target in this notebook** for validation.

> 🇪🇸 Controla:
>
> - **Ventana** = cuántos meses recientes recuerda el modelo;
> - **Horizonte** = cuántos pasos futuros recursivos genera.
>
> Para los primeros 12 pasos existen valores reales reservados.
>
> Después de 12 meses, el pronóstico sale del conjunto disponible y **no existe un valor real en este cuaderno** para validarlo.

In [ ]:
window_slider = widgets.IntSlider(
    value=12,
    min=3,
    max=24,
    step=1,
    description="Window / Ventana:",
    continuous_update=False,
    style={"description_width": "115px"},
)

horizon_slider = widgets.IntSlider(
    value=12,
    min=6,
    max=36,
    step=6,
    description="Horizon / Horizonte:",
    continuous_update=False,
    style={"description_width": "125px"},
)

def explore_recursive_forecast(window, horizon):
    train = y[:-12]

    w_ar = fit_ar(
        train,
        window,
    )

    recursive_pred = recursive_forecast(
        train,
        w_ar,
        window,
        horizon,
    )

    train_end = len(train)

    context_start = max(
        0,
        train_end - 36,
    )

    future_idx = np.arange(
        train_end,
        train_end + horizon,
    )

    available_actual = y[
        train_end:
        min(train_end + horizon, len(y))
    ]

    actual_idx = np.arange(
        train_end,
        train_end + len(available_actual),
    )

    fig, ax = plt.subplots(
        figsize=(10.5, 4.2),
        constrained_layout=True,
    )

    ax.plot(
        np.arange(context_start, train_end),
        y[context_start:train_end],
        marker="o",
        label="training context / contexto de entrenamiento",
    )

    if len(available_actual):
        ax.plot(
            actual_idx,
            available_actual,
            marker="o",
            label="held-out actual / real reservado",
        )

    ax.plot(
        future_idx,
        recursive_pred,
        marker="o",
        label="recursive forecast / pronóstico recursivo",
    )

    ax.axvline(
        train_end - 0.5,
        linestyle="--",
    )

    ax.set_xlabel("month index / índice mensual")
    ax.set_ylabel("passengers / pasajeros")
    ax.set_title(
        f"Recursive forecast / Pronóstico recursivo — "
        f"window={window}, horizon={horizon}"
    )
    ax.legend()

    plt.show()

    comparable = min(
        len(available_actual),
        len(recursive_pred),
    )

    if comparable:
        err = mape(
            available_actual[:comparable],
            recursive_pred[:comparable],
        )

        print(
            f"MAPE on {comparable} real held-out months / "
            f"MAPE en {comparable} meses reales reservados: {err:.1%}"
        )

    print()
    print("EN: every predicted month becomes an input to the next prediction.")
    print("ES: cada mes predicho se convierte en entrada de la siguiente predicción.")

    if horizon > 12:
        print("EN: after step 12, this notebook has no real future target for validation.")
        print("ES: después del paso 12, este cuaderno no tiene un valor futuro real para validar.")

forecast_output = widgets.interactive_output(
    explore_recursive_forecast,
    {
        "window": window_slider,
        "horizon": horizon_slider,
    },
)

display(
    widgets.VBox([
        widgets.HBox([
            window_slider,
            horizon_slider,
        ]),
        forecast_output,
    ])
)

### Recursive feedback vs one-step diagnostic / Retroalimentación recursiva vs diagnóstico de un paso

This comparison isolates an important idea.

### Recursive forecast
After predicting January, the predicted January value is used to predict February.

### One-step diagnostic
For each held-out month, use the **real previous values** to predict only the next month.

This second method is useful as a diagnostic, but it is **not** the same deployment scenario because it uses held-out observations as lag inputs.

If recursive error is noticeably larger, feedback is part of the problem.

> 🇪🇸 Esta comparación separa dos fuentes de error.
>
> En el pronóstico recursivo, una predicción entra en la siguiente predicción.
>
> En el diagnóstico de un paso, cada mes reservado se predice usando valores previos reales. Es útil para diagnosticar, pero no representa el mismo escenario de despliegue.

In [ ]:
feedback_window = widgets.IntSlider(
    value=12,
    min=3,
    max=24,
    step=1,
    description="Window / Ventana:",
    continuous_update=False,
    style={"description_width": "115px"},
)

def compare_feedback(window):
    train = y[:-12]
    actual = y[-12:]

    w_ar = fit_ar(
        train,
        window,
    )

    recursive_pred = recursive_forecast(
        train,
        w_ar,
        window,
        12,
    )

    one_step_pred = one_step_diagnostic(
        y,
        len(train),
        w_ar,
        window,
        12,
    )

    recursive_err = mape(
        actual,
        recursive_pred,
    )

    one_step_err = mape(
        actual,
        one_step_pred,
    )

    x = np.arange(12)

    fig, ax = plt.subplots(
        figsize=(9.5, 4.0),
        constrained_layout=True,
    )

    ax.plot(
        x,
        actual,
        marker="o",
        label="actual / real",
    )

    ax.plot(
        x,
        recursive_pred,
        marker="o",
        label="recursive / recursivo",
    )

    ax.plot(
        x,
        one_step_pred,
        marker="o",
        label="one-step diagnostic / diagnóstico un paso",
    )

    ax.set_xlabel("held-out month / mes reservado")
    ax.set_ylabel("passengers / pasajeros")
    ax.set_title(
        f"Feedback diagnostic / Diagnóstico de retroalimentación — "
        f"window={window}"
    )
    ax.legend()

    plt.show()

    print(
        "Recursive MAPE / MAPE recursivo:",
        f"{recursive_err:.1%}",
    )

    print(
        "One-step diagnostic MAPE / MAPE diagnóstico un paso:",
        f"{one_step_err:.1%}",
    )

    print()
    print("EN: the one-step line uses real previous held-out values; use it only to diagnose recursive feedback.")
    print("ES: la línea de un paso usa valores reales previos del periodo reservado; úsala solo para diagnosticar la retroalimentación recursiva.")

feedback_output = widgets.interactive_output(
    compare_feedback,
    {"window": feedback_window},
)

display(
    widgets.VBox([
        feedback_window,
        feedback_output,
    ])
)

### Follow the feedback loop one step at a time / Sigue el ciclo de retroalimentación paso a paso

Choose the forecast step.

The notebook shows the most recent values used by the model and whether those inputs are:

- real training observations;
- earlier recursive predictions.

> 🇪🇸 Elige el paso del pronóstico.
>
> El cuaderno muestra los valores recientes usados por el modelo e indica si provienen de observaciones reales de entrenamiento o de predicciones recursivas anteriores.

In [ ]:
feedback_step = widgets.IntSlider(
    value=1,
    min=1,
    max=12,
    step=1,
    description="Forecast step / Paso:",
    continuous_update=False,
    style={"description_width": "135px"},
)

def inspect_recursive_step(step):
    p = 12

    train = y[:-12]
    w_ar = fit_ar(train, p)

    hist = list(train.astype(float))
    predictions = []

    used_values = None

    for current_step in range(1, step + 1):
        used_values = np.asarray(
            hist[-p:],
            dtype=float,
        )

        nxt = float(
            w_ar[0]
            + np.dot(
                w_ar[1:],
                used_values,
            )
        )

        predictions.append(nxt)
        hist.append(nxt)

    n_pred_inputs = max(
        0,
        step - 1,
    )

    n_real_inputs = p - min(
        p,
        n_pred_inputs,
    )

    print("Forecast step / Paso:", step)
    print("Window / Ventana:", p)
    print("Real inputs in current window / Entradas reales:", n_real_inputs)
    print("Predicted inputs in current window / Entradas predichas:", min(p, n_pred_inputs))
    print()
    print("Recent values used / Valores recientes usados:")
    print(np.round(used_values, 1).tolist())
    print()
    print("Prediction / Predicción:", round(predictions[-1], 1))

    if step == 1:
        print("EN: every lag value is still a real training observation.")
        print("ES: todos los valores rezagados siguen siendo observaciones reales de entrenamiento.")
    else:
        print("EN: earlier model predictions have begun to enter the next prediction.")
        print("ES: predicciones anteriores del modelo ya empezaron a entrar en la siguiente predicción.")

feedback_step_output = widgets.interactive_output(
    inspect_recursive_step,
    {"step": feedback_step},
)

display(
    widgets.VBox([
        feedback_step,
        feedback_step_output,
    ])
)

<details>
<summary><strong>Why can recursive error grow? / ¿Por qué puede crecer el error recursivo?</strong></summary>

Suppose the first forecast is a little too high.

The next forecast uses that slightly high value as part of its input.

If the model keeps reinforcing that mistake, the error can travel through later steps.

This does **not** mean recursive models always diverge.

It means long-horizon behaviour depends on:

- the fitted coefficients;
- the data dynamics;
- the forecast horizon;
- how much predicted information re-enters the state.

> 🇪🇸 Si la primera predicción es un poco alta, la siguiente puede usar ese valor alto como entrada.
>
> El error puede propagarse, aunque eso no significa que todo modelo recursivo necesariamente diverja.
>
> El comportamiento depende de los coeficientes, la dinámica de los datos, el horizonte y cuánta información predicha vuelve a entrar en el estado.

</details>

## Bridge to recurrent neural networks / Puente hacia redes neuronales recurrentes

The recurrence:

`x[t+1] = A @ x[t]`

is not itself an RNN.

But it gives the key mental model:

> **reuse parameters across time while carrying state forward.**

A recurrent neural network adds learned nonlinear transformations and a hidden state, but the repeated state-update idea remains.

> 🇪🇸 La recurrencia `x[t+1] = A @ x[t]` no es por sí sola una RNN.
>
> Pero entrega la idea mental central:
>
> **reutilizar parámetros a través del tiempo mientras un estado transporta información hacia adelante.**

## Quick reasoning challenge / Reto rápido de razonamiento

Choose a situation and identify what repetition is doing.

> 🇪🇸 Elige una situación e identifica qué está haciendo la repetición.

In [ ]:
recursion_case = widgets.Dropdown(
    options=[
        ("Fibonacci", "fib"),
        ("Power iteration / Iteración de potencias", "power"),
        ("Recursive forecasting / Pronóstico recursivo", "forecast"),
    ],
    value="fib",
    description="Case / Caso:",
    style={"description_width": "90px"},
)

def explain_recursion_case(case):
    answers = {
        "fib": (
            "Repeated updates generate the exact sequence state.",
            "Las actualizaciones repetidas generan exactamente el estado de la secuencia.",
        ),
        "power": (
            "Repeated multiplication amplifies the dominant eigendirection.",
            "La multiplicación repetida amplifica la dirección propia dominante.",
        ),
        "forecast": (
            "Repeated prediction feeds model output back into future model input.",
            "La predicción repetida devuelve la salida del modelo como entrada futura.",
        ),
    }

    en, es = answers[case]

    print("EN:", en)
    print("ES:", es)

recursion_case_output = widgets.interactive_output(
    explain_recursion_case,
    {"case": recursion_case},
)

display(
    widgets.VBox([
        recursion_case,
        recursion_case_output,
    ])
)

## What just happened / Qué acaba de pasar

You used one idea in three different settings:

> **current state → update rule → next state → repeat**

### 1. Fibonacci

State:

`[f[n], f[n-1]]`

The same matrix update generated the sequence.

### 2. Power iteration

State:

`current direction`

Repeated multiplication plus normalization moved the state toward the dominant eigendirection.

When `|λ₂/λ₁|` approached `1`, convergence became slower.

### 3. Real airline forecasting

State:

`recent passenger history`

The pseudoinverse fitted a one-step linear update from real observations.

Recursive forecasting then reused predictions as future inputs.

That made **feedback** part of the forecasting problem.

### Four ideas to remember / Cuatro ideas para recordar

1. **State is memory. / El estado es memoria.**
2. **A recurrence reuses an update rule. / Una recurrencia reutiliza una regla de actualización.**
3. **Repeated multiplication can amplify a direction. / La multiplicación repetida puede amplificar una dirección.**
4. **Repeated prediction can propagate error. / La predicción repetida puede propagar error.**

### The sentence to remember / La frase para recordar

> **Recursion is repeated state update: the next input contains information produced by the previous step.**

> 🇪🇸
>
> **La recursión es una actualización repetida del estado: la siguiente entrada contiene información producida por el paso anterior.**

### Final self-check / Autoevaluación final

If a model predicts 12 months recursively, what changes after the first predicted month?

**The model is no longer using only observed history — its own predictions begin entering the state.**

> 🇪🇸 Si un modelo pronostica 12 meses recursivamente, ¿qué cambia después del primer mes predicho?
>
> **El modelo deja de usar únicamente historia observada: sus propias predicciones empiezan a entrar en el estado.**

---

## Done with this section / Fin de esta sección

Next / Siguiente: **09 · Convolution and deconvolution / Convolución y deconvolución** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/09-convolution-and-deconvolution.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)